<h2>Defining context using @dataclass </h2>

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from dataclasses import dataclass

@dataclass
class fav_food:
    favourite_breakfast: str ="dosa"
    favourite_lunch: str ="south indian Tali"
    favourite_dinner:str ="Chepati panner"


In [11]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

model=ChatGoogleGenerativeAI(model="gemini-3.6-flash")
agent= create_agent(
    model=model,
    context_schema=fav_food
)

In [12]:
from langchain.messages import HumanMessage

response= agent.invoke(
    {"messages":[HumanMessage(content="what is my favourite breakfast")]},
    context=fav_food()
)

In [5]:
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content='what is my favourite breakfast', additional_kwargs={}, response_metadata={}, id='25e8075e-8143-48ef-bebe-567767813616'),
              AIMessage(content=[{'type': 'text', 'text': "I don't know your favorite breakfast yet! Since I don't have access to your personal information, you'll have to tell me. \n\nIs it something sweet like pancakes or waffles, something savory like eggs and bacon, or maybe a quick smoothie? What's your go-to morning meal?", 'extras': {'signature': 'EsUJCsIJAWkUfRN19plTNZEYJbCA+PGNCVNoWh1jhvCangTOl7LpxS8yvWyiqiQ9UbaD1wgFypsDG5dgWnjRcI6P7foYDzMzQB3QDXprZ/mGdvuG5Ua0ikkIqRykjhD2PKD5oBSqild5Vqexfw4q9n0cD/HZYAB1/YURojVNIqToIkgqH/IDw+ARAce2fCK/9YpFwCj0aDS3RG0R2X2vwxpE442tPdHkkySTxdbxipExvVmjLx33MURFaMrhKpz4MWEQYG0rLd+muNU8dCJCk/xqb+A7QidXkTjIspSbBNgyYvs1tkn18DBuEdlQGzlYX9v9K2t7agijv3NgXifUGQ6e8s2bTsycY8Bu/pQE3X3m1j75aEQogTGxf3D2Knp3bTuYXsJifdbSTx9Q9zBhozq05VUvtlvY/4XFV9gaF9g99tqgWMW1siMDl/vpk7rt3DyFI/EbdLj8/lO0Q7eB5eEfAU6G3EFGWHnfR9

<h2>Accessing Context</h2>
<p> By default context can't be accessed by agent through tool calls only it will be accessed</p>

In [13]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_breakfast(runtime: ToolRuntime[fav_food])-> str:
    """get fav breakfast of the user"""
    return runtime.context.favourite_breakfast

@tool
def get_favourite_lunch(runtime: ToolRuntime[fav_food])-> str:
    """get fav breakfast of the user"""
    return runtime.context.favourite_lunch

@tool
def get_favourite_dinner(runtime: ToolRuntime[fav_food])-> str:
    """get fav breakfast of the user"""
    return runtime.context.favourite_dinner

In [14]:
agent= create_agent(
    model=model,
    tools=[get_favourite_breakfast,get_favourite_dinner,get_favourite_lunch],
    context_schema=fav_food
)

In [19]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite breakfast,lunch,dinner?")]},
    context=fav_food()
)

# pprint(response)

print(response['messages'][-1].content[0]['text'])

Your favourite meals are:

- **Breakfast:** Dosa
- **Lunch:** South Indian Thali
- **Dinner:** Chapati Paneer


In [20]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite breakfast?")]},
    context=fav_food(favourite_breakfast="idly")
)

# pprint(response)

print(response['messages'][-1].content[0]['text'])

Your favourite breakfast is idly.
